In [3]:
import os
import requests

In [49]:
base_url = "https://api.panoramax.xyz"
picture_id ="6058367b-b429-4f34-8955-674ace44c18b"

output_dir = "streetview_images"
output_filename = f"{picture_id}.jpg"
output_path = os.path.join(output_dir, output_filename)

def download_image(picture_id):
    try:
        path = f"api/pictures/{picture_id}"
        metadata_url = f"{base_url}/{path}"
        os.makedirs(output_dir, exist_ok=True)
        print(f"Directory verified: {output_dir}")

        # Get the picture metadata (JSON response)
        print(f"Fetching metadata from: {metadata_url}")
        response = requests.get(metadata_url)
        response.raise_for_status() # Check for HTTP errors
        data = response.json()

        # extract azimuth
        properties = data.get('properties', {})

        # Extract the image URL from the response
        # Panoramax/GeoVisio returns a STAC Item. The image links are in the 'assets' dictionary.
        # We prioritize 'hd' (High Definition) but can fall back to 'sd' (Standard Definition).
        assets = data.get('assets', {})
        
        if 'hd' in assets:
            image_url = assets['hd']['href']
            print("Found HD image URL.")
        elif 'sd' in assets:
            image_url = assets['sd']['href']
            print("HD not available, found SD image URL.")
        else:
            raise ValueError("No suitable image asset ('hd' or 'sd') found in the response.")

        print(f"Image URL extracted: {image_url}")
        print("Downloading image...")
        image_response = requests.get(image_url)
        image_response.raise_for_status()
        with open(output_path, 'wb') as file:
            file.write(image_response.content)
        print(f"Success! Image saved to: {output_path}")
        return properties

    except requests.exceptions.RequestException as e:
        print(f"Network error occurred: {e}")
    except Exception as e:
        print(f"An error occurred: {e}")
    

In [50]:
properties = download_image("afd9aa9b-6737-4809-b3e3-1c2fe87dcddb")

Directory verified: streetview_images
Fetching metadata from: https://api.panoramax.xyz/api/pictures/afd9aa9b-6737-4809-b3e3-1c2fe87dcddb
Found HD image URL.
Image URL extracted: https://panoramax-storage-public-fast.s3.gra.perf.cloud.ovh.net/main-pictures/af/d9/aa/9b/6737-4809-b3e3-1c2fe87dcddb.jpg
Success! Image saved to: streetview_images/6058367b-b429-4f34-8955-674ace44c18b.jpg


In [52]:
azimuth = properties.get('view:azimuth')
azimuth

41

In [57]:
properties.keys()

dict_keys(['exif', 'created', 'license', 'updated', 'datetime', 'semantics', 'collection', 'datetimetz', 'annotations', 'view:azimuth', 'geovisio:image', 'geovisio:status', 'geovisio:producer', 'geovisio:thumbnail', 'original_file:name', 'original_file:size', 'geovisio:visibility', 'pers:interior_orientation', 'geovisio:rank_in_collection', 'quality:horizontal_accuracy', 'panoramax:horizontal_pixel_density'])

In [ ]:
import requests
import math
import os

# --- Configuration ---
BASE_URL = "https://api.panoramax.xyz"
SEARCH_URL = f"{BASE_URL}/api/search"

# Center Point lat:  Center Point lon: 
LATITUDE = 49.17359902913684
LONGITUDE = -0.3482916490346519
SEARCH_RADIUS_METERS = 5 
OUTPUT_DIR = "streetview_360"

def get_bbox_from_point(lat, lon, dist_meters):
    earth_radius = 6378137
    d_lat = dist_meters / earth_radius
    d_lon = dist_meters / (earth_radius * math.cos(math.pi * lat / 180))
    
    # Offsets in degrees
    d_lat_deg = d_lat * 180 / math.pi
    d_lon_deg = d_lon * 180 / math.pi
    
    return f"{lon - d_lon_deg},{lat - d_lat_deg},{lon + d_lon_deg},{lat + d_lat_deg}"

# 1. Prepare
os.makedirs(OUTPUT_DIR, exist_ok=True)
bbox_string = get_bbox_from_point(LATITUDE, LONGITUDE, SEARCH_RADIUS_METERS)
print(f"Searching area: {bbox_string}")

params = {
    "bbox": bbox_string,
    "limit": 100
}

try:
    # 2. Search API
    response = requests.get(SEARCH_URL, params=params)
    response.raise_for_status()
    features = response.json().get("features", [])
    
    print(f"Total images found: {len(features)}")
    
    count_360 = 0
    
    # 3. Filter and Download
    for feature in features:
        props = feature.get("properties", {})
        
        # --- THE FILTER ---
        # We check if the projection type is 'equirectangular'
        projection = props.get("GPano:ProjectionType")
        
        if projection == "equirectangular":
            count_360 += 1
            pid = feature["id"]
            
            # Extract URL (HD)
            assets = feature.get("assets", {})
            img_url = assets.get("hd", {}).get("href") or assets.get("sd", {}).get("href")
            
            if img_url:
                print(f"Downloading 360° Image: {pid}")
                img_data = requests.get(img_url).content
                
                with open(os.path.join(OUTPUT_DIR, f"{pid}.jpg"), 'wb') as f:
                    f.write(img_data)
        else:
            # Skip flat/standard photos
            continue

    print(f"Done. Saved {count_360} panoramic images.")

except Exception as e:
    print(f"Error: {e}")

Searching area: -0.34897867672262883,49.17314987149478,-0.3476046213466749,49.1740481867789
Total images found: 16
Done. Saved 0 panoramic images.
